# Phase C1: EfficientNet-B0 + SED Head (20-sec chunks)
Proper Stage 1 following 2025 1st place architecture.
Changes from C0: 20-sec chunks, SED head, n_mels=224, CrossEntropy loss, MixUp weight=0.5, batch_size=64.

In [1]:
#Cell 2 (Code): Config -- all changes marked with comments

import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
import librosa
import soundfile as sf
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

CFG = {
    'seed': 42,
    'base_dir': '/data/scratch/scienceteam/jupyter-mir/bird/Data/birdclef-2026',
    'output_dir': '/data/scratch/scienceteam/jupyter-mir/bird/models/stage1_sed',
    'sr': 32000,

    # ── Changed: 20-sec chunks (was 5s) ──
    'duration': 20,

    # ── Changed: mel params from 2025 1st place ──
    'n_fft': 4096,
    'hop_length': 1252,
    'n_mels': 224,
    'fmin': 0,
    'fmax': 16000,
    'top_db': 80.0,
    'img_width': 512,        # time frames for 20s: (20*32000)//1252 + 1 ≈ 512

    # ── Changed: proper timm model name ──
    'model_name': 'tf_efficientnet_b0.ns_jft_in1k',
    'num_classes': 234,

    # ── Changed: batch_size 64 (was 32) ──
    'batch_size': 180,
    'epochs': 15,
    'patience': 3,

    # ── Changed: lr from 2025 1st place ──
    'lr': 5e-4,
    'weight_decay': 1e-4,

    # ── Changed: MixUp constant 0.5 (was Beta 0.15) ──
    'mixup_weight': 0.5,

    'n_folds': 5,
    'num_workers': 4,
}

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(CFG['seed'])
os.makedirs(CFG['output_dir'], exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE} ({torch.cuda.get_device_name(0)})")

Device: cuda (NVIDIA A40)


In [2]:
#Cell 3 (Code): Load data

train = pd.read_csv(os.path.join(CFG['base_dir'], 'train.csv'))
taxonomy = pd.read_csv(os.path.join(CFG['base_dir'], 'taxonomy.csv'))

LABELS = sorted(taxonomy['primary_label'].tolist())
LABEL2IDX = {label: idx for idx, label in enumerate(LABELS)}
IDX2LABEL = {idx: label for label, idx in LABEL2IDX.items()}

train['filepath'] = train['filename'].apply(lambda x: os.path.join(CFG['base_dir'], 'train_audio', x))

print(f"Total training samples: {len(train):,}")
print(f"Total target species: {len(LABELS)}")
print(f"Species with training data: {train['primary_label'].nunique()}")

Total training samples: 35,549
Total target species: 234
Species with training data: 206


In [3]:
# Cell 4 (Code): Dataset -- 20-sec chunks, absmax normalization, higher mel params

class BirdDataset(Dataset):
    def __init__(self, df, label2idx, cfg, is_train=True):
        self.df = df.reset_index(drop=True)
        self.label2idx = label2idx
        self.cfg = cfg
        self.is_train = is_train
        self.target_len = cfg['sr'] * cfg['duration']  # 640,000 samples for 20s

    def __len__(self):
        return len(self.df)

    def load_audio(self, filepath):
        try:
            y, sr = librosa.load(filepath, sr=self.cfg['sr'])
        except Exception:
            return np.zeros(self.target_len, dtype=np.float32)

        if len(y) == 0:
            return np.zeros(self.target_len, dtype=np.float32)

        # Tile short clips to fill 20 seconds
        if len(y) < self.target_len:
            reps = (self.target_len // len(y)) + 1
            y = np.tile(y, reps)

        # Extract window
        if self.is_train and len(y) > self.target_len:
            start = random.randint(0, len(y) - self.target_len)
            y = y[start:start + self.target_len]
        else:
            center = len(y) // 2
            half = self.target_len // 2
            y = y[center - half:center - half + self.target_len]

        # Absmax normalization (from 1st place)
        max_val = np.abs(y).max()
        if max_val > 0:
            y = y / max_val

        return y.astype(np.float32)

    def audio_to_melspec(self, y):
        mel = librosa.feature.melspectrogram(
            y=y, sr=self.cfg['sr'],
            n_fft=self.cfg['n_fft'],
            hop_length=self.cfg['hop_length'],
            n_mels=self.cfg['n_mels'],
            fmin=self.cfg['fmin'],
            fmax=self.cfg['fmax'],
        )
        mel_db = librosa.power_to_db(mel, ref=np.max, top_db=self.cfg['top_db'])

        # 0-1 normalization
        mel_db = (mel_db - mel_db.min()) / (mel_db.max() - mel_db.min() + 1e-8)
        return mel_db

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        y = self.load_audio(row['filepath'])
        mel = self.audio_to_melspec(y)

        # Pad/trim to fixed width
        if mel.shape[1] < self.cfg['img_width']:
            mel = np.pad(mel, ((0, 0), (0, self.cfg['img_width'] - mel.shape[1])))
        else:
            mel = mel[:, :self.cfg['img_width']]

        # 3 channels (EfficientNet expects RGB-like input)
        mel = np.stack([mel, mel, mel], axis=0)
        mel = torch.tensor(mel, dtype=torch.float32)

        target = torch.zeros(self.cfg['num_classes'], dtype=torch.float32)
        if row['primary_label'] in self.label2idx:
            target[self.label2idx[row['primary_label']]] = 1.0

        return mel, target

# Quick test
test_ds = BirdDataset(train.head(2), LABEL2IDX, CFG, is_train=True)
mel, target = test_ds[0]
print(f"Mel shape: {mel.shape}")        # [3, 224, 512]
print(f"Target shape: {target.shape}")   # [234]

Mel shape: torch.Size([3, 224, 512])
Target shape: torch.Size([234])


In [4]:
# Cell 5 (Code): SED Model

class GeM(nn.Module):
    """Generalized Mean Pooling."""
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return x.clamp(min=self.eps).pow(self.p).mean(dim=-1).pow(1.0 / self.p)


class SEDModel(nn.Module):
    """Sound Event Detection model with per-frame predictions."""
    def __init__(self, cfg):
        super().__init__()
        self.backbone = timm.create_model(
            cfg['model_name'], pretrained=True,
            in_chans=3, num_classes=0, global_pool=''
        )
        # EfficientNet-B0 always outputs 1280 channels
        self.feat_dim = 1280
        self.freq_pool = GeM(p=3)
        self.fc = nn.Linear(self.feat_dim, cfg['num_classes'])

    def forward(self, x):
        features = self.backbone(x)
        x = self.freq_pool(features)
        x = x.transpose(1, 2)
        frame_logits = self.fc(x)
        clip_logits = frame_logits.mean(dim=1)
        clip_logits_max = frame_logits.max(dim=1)[0]
        output = 0.5 * clip_logits + 0.5 * clip_logits_max
        return output

# Quick test
model = SEDModel(CFG).to(DEVICE)
dummy = torch.randn(2, 3, 224, 512).to(DEVICE)
out = model(dummy)
print(f"Output shape: {out.shape}")
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")
del model, dummy, out
torch.cuda.empty_cache()

Output shape: torch.Size([2, 234])
Total parameters: 4,307,303


In [5]:
# Cell 6 (Code): MixUp -- constant weight 0.5, take max of targets

def mixup(mel, target, weight=0.5):
    """MixUp with constant blending weight (from 2025 1st place).
    Target: take element-wise max (not weighted sum).
    """
    indices = torch.randperm(mel.size(0))
    mel = weight * mel + (1 - weight) * mel[indices]
    target = torch.max(target, target[indices])
    return mel, target


def spec_augment(mel, freq_mask=30, time_mask=50):
    """SpecAugment -- slightly larger masks for 20-sec input."""
    _, n_mels, n_time = mel.shape

    f = random.randint(0, freq_mask)
    f0 = random.randint(0, max(0, n_mels - f))
    mel[:, f0:f0+f, :] = 0

    t = random.randint(0, time_mask)
    t0 = random.randint(0, max(0, n_time - t))
    mel[:, :, t0:t0+t] = 0

    return mel

print("MixUp (constant 0.5, max target) and SpecAugment defined.")

MixUp (constant 0.5, max target) and SpecAugment defined.


In [6]:
# Cell 7 (Code): Training loop -- CrossEntropy loss, CosineAnnealingWarmRestarts

def train_one_epoch(model, loader, optimizer, scheduler, device, cfg):
    model.train()
    losses = []
    criterion = nn.CrossEntropyLoss()

    pbar = tqdm(loader, desc="Train")
    for mel, target in pbar:
        mel, target = mel.to(device), target.to(device)

        # MixUp with constant weight 0.5
        if random.random() < 0.5:
            mel, target = mixup(mel, target, weight=cfg['mixup_weight'])

        # SpecAugment
        for i in range(mel.size(0)):
            if random.random() < 0.5:
                mel[i] = spec_augment(mel[i])

        logits = model(mel)
        loss = criterion(logits, target)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        pbar.set_postfix(loss=f"{np.mean(losses[-50:]):.4f}")

    scheduler.step()
    return np.mean(losses)


def validate(model, loader, device):
    model.eval()
    all_preds = []
    all_targets = []
    losses = []
    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for mel, target in tqdm(loader, desc="Valid"):
            mel, target = mel.to(device), target.to(device)
            logits = model(mel)
            loss = criterion(logits, target)
            losses.append(loss.item())

            preds = torch.sigmoid(logits).cpu().numpy()
            all_preds.append(preds)
            all_targets.append(target.cpu().numpy())

    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)

    aucs = []
    for i in range(all_targets.shape[1]):
        if all_targets[:, i].sum() > 0:
            try:
                auc = roc_auc_score(all_targets[:, i], all_preds[:, i])
                aucs.append(auc)
            except ValueError:
                pass

    macro_auc = np.mean(aucs) if aucs else 0.0
    return np.mean(losses), macro_auc, len(aucs)

print("Training functions defined.")

Training functions defined.


In [7]:
# Cell 8 (Code): GPU memory test first
torch.cuda.reset_peak_memory_stats()
model = SEDModel(CFG).to(DEVICE)
for bs in [32, 64, 128, 200, 212, ]:
    try:
        torch.cuda.empty_cache()
        dummy = torch.randn(bs, 3, 224, 512).to(DEVICE)
        out = model(dummy)
        loss = out.sum()
        loss.backward()
        mem = torch.cuda.max_memory_allocated() / 1024**3
        print(f"Batch size {bs:>4d}: OK  (peak GPU: {mem:.1f} GB / 48 GB)")
        del dummy, out, loss
    except RuntimeError:
        print(f"Batch size {bs:>4d}: OUT OF MEMORY")
        break
del model
torch.cuda.empty_cache()

Batch size   32: OK  (peak GPU: 6.3 GB / 48 GB)
Batch size   64: OK  (peak GPU: 12.5 GB / 48 GB)
Batch size  128: OK  (peak GPU: 25.0 GB / 48 GB)
Batch size  200: OK  (peak GPU: 39.0 GB / 48 GB)
Batch size  212: OK  (peak GPU: 41.3 GB / 48 GB)


In [ ]:
# Cell 9 (Code): Training loop -- 5-fold CV

skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])
fold_results = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train, train['primary_label'])):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}")
    print(f"{'='*60}")

    train_df = train.iloc[train_idx]
    val_df = train.iloc[val_idx]
    print(f"Train: {len(train_df):,} | Val: {len(val_df):,}")

    train_ds = BirdDataset(train_df, LABEL2IDX, CFG, is_train=True)
    val_ds = BirdDataset(val_df, LABEL2IDX, CFG, is_train=False)

    train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                              num_workers=CFG['num_workers'], pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False,
                            num_workers=CFG['num_workers'], pin_memory=True)

    model = SEDModel(CFG).to(DEVICE)
    optimizer = optim.AdamW(model.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, eta_min=1e-6)

    best_auc = 0
    no_improve = 0

    for epoch in range(CFG['epochs']):
        print(f"\n--- Epoch {epoch+1}/{CFG['epochs']} ---")

        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, DEVICE, CFG)
        val_loss, val_auc, n_scored = validate(model, val_loader, DEVICE)

        print(f"Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | Val AUC: {val_auc:.4f} (on {n_scored} species)")

        if val_auc > best_auc:
            best_auc = val_auc
            no_improve = 0
            save_path = os.path.join(CFG['output_dir'], f'c1_fold{fold}.pth')
            torch.save(model.state_dict(), save_path)
            print(f"  -> Saved best model (AUC={best_auc:.4f})")
        else:
            no_improve += 1
            print(f"  -> No improvement ({no_improve}/{CFG['patience']})")
            if no_improve >= CFG['patience']:
                print(f"  -> Early stopping at epoch {epoch+1}")
                break

    fold_results.append({'fold': fold, 'best_auc': best_auc})
    print(f"\nFold {fold} best AUC: {best_auc:.4f}")

    del model, optimizer, scheduler, train_loader, val_loader, train_ds, val_ds
    torch.cuda.empty_cache()

results_df = pd.DataFrame(fold_results)
print(f"\n{'='*60}")
print(f"C1 CROSS-VALIDATION RESULTS")
print(f"{'='*60}")
print(results_df.to_string(index=False))
print(f"\nMean AUC: {results_df['best_auc'].mean():.4f} +/- {results_df['best_auc'].std():.4f}")


FOLD 0
Train: 28,439 | Val: 7,110

--- Epoch 1/15 ---


Valid: 100%|██████████| 40/40 [02:36<00:00,  3.91s/it]


Train loss: 6.3946 | Val loss: 2.3898 | Val AUC: 0.9128 (on 198 species)
  -> Saved best model (AUC=0.9128)

--- Epoch 2/15 ---


Valid: 100%|██████████| 40/40 [02:34<00:00,  3.87s/it]


Train loss: 4.0601 | Val loss: 1.6687 | Val AUC: 0.9516 (on 198 species)
  -> Saved best model (AUC=0.9516)

--- Epoch 3/15 ---


...

Valid: 100%|██████████| 40/40 [03:25<00:00,  5.14s/it]


Train loss: 3.1748 | Val loss: 1.1422 | Val AUC: 0.9707 (on 198 species)
  -> Saved best model (AUC=0.9707)

--- Epoch 8/15 ---


Valid: 100%|██████████| 40/40 [02:48<00:00,  4.22s/it]


Train loss: 2.8091 | Val loss: 1.0630 | Val AUC: 0.9698 (on 198 species)
  -> No improvement (1/3)

--- Epoch 9/15 ---


Valid:  80%|████████  | 32/40 [02:31<00:16,  2.02s/it]

In [ ]:
# Cell 10 (Code): Test predictions

model = SEDModel(CFG).to(DEVICE)
model.load_state_dict(torch.load(os.path.join(CFG['output_dir'], 'c1_fold0.pth'), map_location=DEVICE))
model.eval()

val_df = train.iloc[list(skf.split(train, train['primary_label']))[0][1]]
samples = val_df.sample(5, random_state=42)
val_ds = BirdDataset(samples, LABEL2IDX, CFG, is_train=False)

print("Sample predictions (top 5 species per clip):\n")
for i in range(len(val_ds)):
    mel, target = val_ds[i]
    row = samples.iloc[i]

    with torch.no_grad():
        logits = model(mel.unsqueeze(0).to(DEVICE))
        probs = torch.sigmoid(logits).cpu().numpy()[0]

    top5_idx = probs.argsort()[-5:][::-1]
    true_label = row['primary_label']

    print(f"True: {true_label} ({row['common_name']})")
    for idx in top5_idx:
        marker = " <-- CORRECT" if IDX2LABEL[idx] == true_label else ""
        print(f"  {IDX2LABEL[idx]:>15s}: {probs[idx]:.4f}{marker}")
    print()

del model
torch.cuda.empty_cache()
print("C1 complete!")